# Kaggle Resubmit - archivos faltantes

Reintenta el submit a Kaggle de archivos `.csv` que **ya fueron generados** por `z620_WorkFlow_01_junior_grupoA.ipynb` pero que no se pudieron subir porque se superó el límite diario de submits.

Usa el **mismo nombre de archivo** y el **mismo formato de comentario** que la función `kaggle_submit()` del notebook original (celda 84).

El comentario original se arma con `hyperparams_to_string(params)`, que necesita `num_leaves` / `min_data_in_leaf` / `feature_fraction` / `num_iterations` / `best_auc`. Esos valores no están en el nombre del archivo, pero sí quedaron guardados en `results_summary_<experiment_name>.txt` (los graba `save_experiment_results()`, que en el loop original corre justo después de `kaggle_submit()`, así que si ya tenés el `.csv` generado, lo más probable es que esa fila ya exista).

El script es re-ejecutable: cada submit exitoso queda grabado en un archivo de estado, así que si Kaggle te frena por límite diario (se detecta y corta solo) podés volver a correr la celda del loop principal otro día y sigue donde quedó, sin repetir subidas.

## 1) Librerías

In [1]:
suppressMessages({
  require("data.table")
})


## 2) Configuración — **AJUSTAR ESTO A TU ENTORNO**

- `carpeta_experimento`: la ruta real donde están tus `.csv`, tu `results_summary_<experimento>.txt` y donde `kaggle_submit()` creaba la subcarpeta `kaggle/`.
- `experimento`: el número que aparece en el nombre de archivo (`KA9105_...`).
- `dry_run <- TRUE`: mientras esté en `TRUE`, el loop principal solo **muestra** qué comando ejecutaría, sin subir nada de verdad. Pasalo a `FALSE` recién cuando ya revisaste el listado.

In [2]:
PARAM <- list()
PARAM$carpeta_experimento <- "/content/buckets/b1/exp/WF9105"   # <-- AJUSTAR

# Me paro en la carpeta del experimento, igual que hacia el notebook
# original con setwd(paste0("/content/buckets/b1/exp/", experimento_folder)),
# para que la ruta relativa "./kaggle/..." (igual a la que usa
# kaggle_submit() en el workflow original) resuelva correctamente.
setwd(PARAM$carpeta_experimento)

# Numero de experimento usado en el nombre de archivo (KA9105_...)
PARAM$experimento <- 9105                                        # <-- AJUSTAR si hace falta

# Competencia de Kaggle (la misma que en el notebook original)
PARAM$kaggle_competencia <- "utn-2026-virtual-jr"

# Segundos de espera entre cada submit (subir esto si Kaggle
# se queja de "too many requests"; bajarlo si tenes margen)
PARAM$delay_segundos <- 30

# Si TRUE: solo MUESTRA que haria, no ejecuta ningun submit real.
# Poner en FALSE cuando ya revisaste el listado y queres subir de verdad.
PARAM$dry_run <- FALSE

# Archivo de estado: registra que combinaciones ya fueron
# submiteadas exitosamente, para no repetirlas en una proxima corrida.
PARAM$archivo_estado <- file.path(PARAM$carpeta_experimento, "kaggle_resubmit_estado.csv")

# Archivo de log de esta corrida
PARAM$archivo_log <- file.path(PARAM$carpeta_experimento,
  sprintf("kaggle_resubmit_log_%s.txt", format(Sys.time(), "%Y%m%d_%H%M%S")))


## 3) Combinaciones faltantes (experimento x semilla x corte)

Tal cual las especificaste.

In [3]:
cortes_estandar <- seq(1800, 2400, by = 100)

combos_faltantes <- rbindlist(list(
  CJ(experiment_name = "selective",
     semilla = c(777199),
     corte   = cortes_estandar)
))

cat(sprintf("Total de combinaciones a revisar: %d\n", nrow(combos_faltantes)))


Total de combinaciones a revisar: 7


## 4) Funciones auxiliares

### 4.1 Logging

In [4]:
log_msg <- function(...) {
  msg <- sprintf(...)
  linea <- sprintf("[%s] %s", format(Sys.time(), "%Y-%m-%d %H:%M:%S"), msg)
  cat(linea, "\n")
  cat(linea, "\n", file = PARAM$archivo_log, append = TRUE)
  flush.console()
}


### 4.2 Reconstrucción del string de hiperparámetros
Idéntico a `hyperparams_to_string()` del notebook original (celda 87).

In [5]:
hyperparams_to_string <- function(params) {
  sprintf(
    "num_leaves=%d, min_data_in_leaf=%d, feature_fraction=%.2f, num_iterations=%d - AUC = %.10f",
    params$num_leaves,
    params$min_data_in_leaf,
    params$feature_fraction,
    params$num_iterations,
    params$best_auc
  )
}


### 4.3 Búsqueda de hiperparámetros por experimento + semilla
Lee `results_summary_<experiment_name>.txt`.

In [6]:
buscar_hiperparametros <- function(experiment_name, semilla) {
  archivo <- file.path(PARAM$carpeta_experimento,
    sprintf("results_summary_%s.txt", experiment_name))

  if (!file.exists(archivo)) {
    return(NULL)
  }

  tb <- tryCatch(fread(archivo), error = function(e) NULL)
  if (is.null(tb) || nrow(tb) == 0) return(NULL)

  filas <- tb[seed == semilla]
  if (nrow(filas) == 0) return(NULL)

  if (nrow(filas) > 1) {
    log_msg("AVISO: %d filas encontradas para experimento=%s semilla=%d en %s. Uso la ultima.",
      nrow(filas), experiment_name, semilla, basename(archivo))
    filas <- filas[.N]
  }

  list(
    num_leaves       = filas$num_leaves,
    min_data_in_leaf = filas$min_data_in_leaf,
    feature_fraction = filas$feature_fraction,
    num_iterations   = filas$num_iterations,
    best_auc         = filas$best_auc
  )
}


### 4.4 Archivo de estado (para poder cortar y retomar otro día)

In [7]:
cargar_estado <- function() {
  if (file.exists(PARAM$archivo_estado)) {
    return(fread(PARAM$archivo_estado))
  }
  data.table(
    experiment_name = character(),
    semilla = integer(),
    corte = integer(),
    archivo = character(),
    timestamp = character(),
    resultado = character()
  )
}

guardar_estado <- function(estado, experiment_name, semilla, corte, archivo, resultado) {
  nueva_fila <- data.table(
    experiment_name = experiment_name,
    semilla = semilla,
    corte = corte,
    archivo = archivo,
    timestamp = format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
    resultado = resultado
  )
  estado <- rbindlist(list(estado, nueva_fila))
  fwrite(estado, PARAM$archivo_estado)
  estado
}

ya_submiteado <- function(estado, p_experiment_name, p_semilla, p_corte) {
  # Nombres de parametro distintos a los de las columnas de "estado"
  # (experiment_name / semilla / corte) para evitar el error
  # "object '..corte' not found" que tira el prefijo ".." en
  # algunas versiones de data.table cuando hay choque de nombres.
  nrow(estado[experiment_name == p_experiment_name &
              semilla == p_semilla &
              corte == p_corte &
              resultado == "SUBMITTED"]) > 0
}


### 4.5 Submit de un archivo puntual
Igual que `kaggle_submit()` del notebook original (celda 84): arma el comando, lo ejecuta, imprime/loguea la salida tal cual, espera `delay_segundos`, y listo — sin intentar adivinar si el submit tuvo éxito o no. Ese diagnóstico lo hacés vos mirando el log, igual que siempre.

In [8]:
submit_archivo <- function(experiment_name, semilla, corte, archivo_kaggle, hp) {

  mensaje <- sprintf(
    "-m 'experimento=%s envios=%d semilla=%d \n\nParametros:\n %s'",
    experiment_name,
    corte,
    semilla,
    hyperparams_to_string(hp)
  )

  kaggle   <- file.path(Sys.getenv("HOME"), ".venv", "bin", "kaggle")
  comando  <- paste(shQuote(kaggle), "competitions submit")
  competencia <- paste("-c", PARAM$kaggle_competencia)
  arch     <- paste("-f", archivo_kaggle)
  linea    <- paste(comando, competencia, arch, mensaje)

  if (PARAM$dry_run) {
    log_msg("[DRY-RUN] %s", linea)
    return(invisible(NULL))
  }

  # Igual que el kaggle_submit() original: ejecuto, imprimo la
  # salida tal cual, espero, y sigo. No intento inferir si fue
  # exito o error -- eso se revisa mirando el log / Kaggle.
  salida <- system(linea, intern = TRUE) # el submit a Kaggle
  cat(salida, "\n")
  flush.console()
  log_msg("Salida kaggle: %s", paste(salida, collapse = " | "))
  Sys.sleep(PARAM$delay_segundos)

  invisible(NULL)
}


## 5) Cargar estado previo
Corré esta celda cada vez que retomes (aunque sea el mismo día), para que el loop sepa qué ya está hecho.

In [9]:
estado <- cargar_estado()
log_msg("=== Inicio de corrida. dry_run=%s ===", PARAM$dry_run)
nrow(estado)


[2026-09-13 01:24:13] === Inicio de corrida. dry_run=FALSE === 


[1] 22

## 6) Loop principal — el que hace los submits

**Importante:** con `dry_run <- TRUE` (celda 2) esta celda no sube nada, solo imprime los comandos que ejecutaría. Revisalo antes de poner `dry_run <- FALSE` y volver a correr esta celda.

Cada combinación intentada (haya salido bien o mal) queda marcada como `SUBMITTED` en el archivo de estado, así en una próxima corrida no se repite el envío. Si algo falló, lo vas a ver en el log de todos modos — revisalo y, si hace falta reintentar esa combinación puntual, borrá su fila del archivo de estado.

In [10]:
intentados <- 0
saltados_sin_archivo <- 0
saltados_ya_hechos   <- 0
saltados_sin_hp      <- 0

for (i in seq_len(nrow(combos_faltantes))) {
  exp_name <- combos_faltantes$experiment_name[i]
  semilla  <- combos_faltantes$semilla[i]
  corte    <- combos_faltantes$corte[i]

  if (ya_submiteado(estado, exp_name, semilla, corte)) {
    saltados_ya_hechos <- saltados_ya_hechos + 1
    next
  }

  # Mismo formato que en kaggle_submit() del workflow original:
  # "./kaggle/KA9100_lags_delta-46890_1800.csv" (ruta relativa a
  # PARAM$carpeta_experimento, donde ya nos paramos con setwd)
  archivo_kaggle <- sprintf(
    "./kaggle/KA%d_%s-%d_%d.csv",
    PARAM$experimento, exp_name, semilla, corte
  )

  if (!file.exists(archivo_kaggle)) {
    log_msg("SIN ARCHIVO -> %s (no existe, se saltea)", archivo_kaggle)
    saltados_sin_archivo <- saltados_sin_archivo + 1
    next
  }

  hp <- buscar_hiperparametros(exp_name, semilla)
  if (is.null(hp)) {
    log_msg("SIN HIPERPARAMETROS -> experimento=%s semilla=%d. No encontre fila en results_summary_%s.txt. Se saltea (revisar manualmente).",
      exp_name, semilla, exp_name)
    saltados_sin_hp <- saltados_sin_hp + 1
    next
  }

  log_msg("Submiteando: experimento=%s semilla=%d corte=%d archivo=%s",
    exp_name, semilla, corte, basename(archivo_kaggle))

  submit_archivo(exp_name, semilla, corte, archivo_kaggle, hp)

  if (!PARAM$dry_run) {
    estado <- guardar_estado(estado, exp_name, semilla, corte, archivo_kaggle, "SUBMITTED")
    intentados <- intentados + 1
  }
}


[2026-09-13 01:24:14] Submiteando: experimento=selective semilla=777199 corte=1900 archivo=KA9105_selective-777199_1900.csv 
29 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
[2026-09-13 01:24:16] Salida kaggle: 29 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
[2026-09-13 01:24:46] Submiteando: experimento=selective semilla=777199 corte=2000 archivo=KA9105_selective-777199_2000.csv 
28 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
[2026-09-13 01:24:49] Salida kaggle: 28 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
[2026-09-13 01:25:19] Submiteando: experimento=selective semilla=777199 corte=2100 archivo=KA9105_selective-777199_2100.csv 
27 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
[2026-09-13 01:25:21] Salida kaggle: 27 submissions remaining today. | Successfully submitted to UTN 2026 virtual jr 
[2026-09-13 01:25:51] Submiteando:

## 7) Resumen de la corrida

In [11]:
log_msg("=== Fin de corrida ===")
log_msg("Intentados ahora: %d", intentados)
log_msg("Ya estaban intentados antes (saltados): %d", saltados_ya_hechos)
log_msg("Sin archivo en disco (saltados): %d", saltados_sin_archivo)
log_msg("Sin hiperparametros encontrados (saltados, revisar): %d", saltados_sin_hp)
log_msg("Revisa el log completo para ver que submits salieron bien y cuales no: %s", PARAM$archivo_log)
log_msg("Estado completo guardado en: %s", PARAM$archivo_estado)


[2026-09-13 01:27:29] === Fin de corrida === 
[2026-09-13 01:27:29] Intentados ahora: 6 
[2026-09-13 01:27:29] Ya estaban intentados antes (saltados): 1 
[2026-09-13 01:27:29] Sin archivo en disco (saltados): 0 
[2026-09-13 01:27:29] Sin hiperparametros encontrados (saltados, revisar): 0 
[2026-09-13 01:27:29] Revisa el log completo para ver que submits salieron bien y cuales no: /content/buckets/b1/exp/WF9105/kaggle_resubmit_log_20260913_012409.txt 
[2026-09-13 01:27:29] Estado completo guardado en: /content/buckets/b1/exp/WF9105/kaggle_resubmit_estado.csv 
